# <span style="color:rgb(32, 33, 34);font-family:Lato, &quot;Lucida Sans Unicode&quot;, &quot;Lucida Grande&quot;, sans-serif;font-size:40px;letter-spacing:0.2px;background-color:rgb(255, 255, 255);">HW 8 - Chapter 7: T-SQL for data analysis</span>

## Proposition #1

Find the value (subtotal) of each order that each salesperson managed to make a sale for and also return the running total of the amount that each salesperson has earned. This is to see how much each salesperson has earned, both from each order and in total. 

(use Sales.SalesOrderHeader and Person.Person<span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">)</span>

In [ ]:
USE AdventureWorks2019;

SELECT CONCAT(P.FirstName, ' ', P.MiddleName, ' ', P.LastName) as FullName,
       H.SalesPersonID, 
       H.SalesOrderID, 
       H.SubTotal, 
       SUM(H.SubTotal) OVER(PARTITION BY H.SalesPersonID ORDER BY H.SalesPersonID
                            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as RunningTotal,
       ROW_NUMBER()    OVER(PARTITION BY H.SalesPersonID ORDER BY H.SalesPersonID) as NumofOrders

FROM Sales.SalesOrderHeader as H, Person.Person as P
WHERE H.SalesPersonID IS NOT NULL AND
      P.BusinessEntityID = H.SalesPersonID

## Proposition #2

Find the dense rank of each customer's orders of each type of product that they've ordered. This is to how many distinct products that they've ordered. (note that customers under PersonType are registered under IN, Individual \[retail\] customer)

(use Person.Person and Sales.SalesOrderDetail)

In [ ]:
USE AdventureWorks2019;

SELECT CONCAT(P.FirstName, ' ', P.MiddleName, ' ', P.LastName) as FullName,
       H.CustomerID,
       H.SalesOrderID,
       D.ProductID, 
       DENSE_RANK() OVER (ORDER BY D.ProductID) as DistinctNumofProducts    -- get the distinct number of products ordered, 
                                                                            -- organized by ProductID

FROM Person.Person as P, Sales.SalesOrderDetail as D, Sales.SalesOrderHeader as H
WHERE P.PersonType = 'IN' AND                   -- where each person is a customer, and
      P.BusinessEntityID = H.CustomerID AND     -- is a customer that has made an order, and
      H.SalesOrderID = D.SalesOrderID           -- find their orders in both OrderDetail and OrderHeader
ORDER BY D.ProductId    -- visually order by ProductID

## Proposition #3

Find the running sum of costs up to each customer's subsequent order, ordered by increasing in cost. This is to see how much they've spent before in total. <span style="font-family: -apple-system, BlinkMacSystemFont, sans-serif; color: var(--vscode-foreground);">(note that customers under PersonType are registered under IN, Individual [retail] customer)</span>

(use Person.Person, Sales.SalesOrderDetail, and Sales.SalesOrderHeader)

In [ ]:
USE AdventureWorks2019;

SELECT CONCAT(P.FirstName, ' ', P.MiddleName, ' ', P.LastName) as FullName,
       H.CustomerID,
       H.SalesOrderID,
       H.TotalDue as CurrentCost,
       SUM(H.TotalDue) OVER(ORDER BY H.CustomerID ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as RunningTotalCosts

FROM Person.Person as P, Sales.SalesOrderDetail as D, Sales.SalesOrderHeader as H
WHERE P.PersonType = 'IN' AND                   -- where each person is a customer, and
      P.BusinessEntityID = H.CustomerID AND     -- is a customer that has made an order, and
      H.SalesOrderID = D.SalesOrderID           -- find their orders in both OrderDetail and OrderHeader

## Proposition #4

Find how many distinct salespeople that each customer has ordered from with each subsequent order. This is to see how many distinct salespeople have convinced each customer to buy something. With how so many customers in the query result only have 1 distinct salespeople whose IDs are NULL, it means that they don't order products from any salespeople registered in AdventureWorks. <span style="color: var(--vscode-foreground); font-family: -apple-system, BlinkMacSystemFont, sans-serif;">(note that customers under PersonType are registered under IN, Individual [retail] customer)</span>

(use Person.Person and Sales.SalesOrderHeader)

In [ ]:
USE AdventureWorks2019;

SELECT CONCAT(P.FirstName, ' ', P.MiddleName, ' ', P.LastName) as FullName,
       H.CustomerID,
       H.SalesPersonID,
       DENSE_RANK() OVER (PARTITION BY H.CustomerID ORDER BY H.SalesPersonID) as DistinctSalespeople -- get the distinct number of salespeople involved in each subsequent order 

FROM Person.Person as P, Sales.SalesOrderHeader as H
WHERE P.PersonType = 'IN' AND                   -- where each person is a customer, and
      P.BusinessEntityID = H.CustomerID         -- is a customer that has made an order

## Proposition #5

Find the last time that each product was ordered prior to each order. This is to see when the last time a particular product was ordered and to get a better idea of how frequently each type of product was ordered.

(use Sales.SalesOrderDetail and Sales.SalesOrderHeader)

In [ ]:
USE AdventureWorks2019;

SELECT D.ProductID,
       H.OrderDate as CurrentDate,
       LAG(H.OrderDate) OVER(PARTITION BY D.ProductID ORDER BY H.OrderDate) as PreviousDate -- get the previous order date with LAG
FROM Sales.SalesOrderDetail as D, Sales.SalesOrderHeader as H
WHERE D.SalesOrderID = H.SalesOrderID

## Proposition #6

Find the first and last time that each customer ordered something from AdventureWorks, as well as track how many times they've ordered something. This is to see how long they've been ordering products through the manufacturer as well as their total orders throughout it all.

(use Person.Person, Sales.SalesOrderDetail, and Sales.SalesOrderHeader)

In [ ]:
USE AdventureWorks2019;

SELECT CONCAT(P.FirstName, ' ', P.MiddleName, ' ', P.LastName) as FullName,
       H.CustomerID,
       H.SalesOrderID,
       FIRST_VALUE(H.OrderDate) OVER(PARTITION BY H.CustomerID ORDER BY H.OrderDate) as FirstOrderDate,     -- the first order date
       LAST_VALUE(H.OrderDate) OVER(PARTITION BY H.CustomerID ORDER BY H.OrderDate) as LastOrderDate,       -- the last order date
       ROW_NUMBER() OVER(PARTITION BY H.CustomerID ORDER BY H.OrderDate) as RunningOrders

FROM Person.Person as P, Sales.SalesOrderDetail as D, Sales.SalesOrderHeader as H
WHERE P.PersonType = 'IN' AND                   -- where each person is a customer, and
      P.BusinessEntityID = H.CustomerID AND     -- is a customer that has made an order, and
      H.SalesOrderID = D.SalesOrderID           -- find their orders in both OrderDetail and OrderHeader

## Proposition #7

Find the last time that each transaction's product type was involved in a transaction of each product type. This is to see how often transactions are made around a particular product.

(use Production.TransactionHistory)

In [ ]:
USE AdventureWorks2019;

SELECT ProductID,
       TransactionDate as CurrentTransDate,
       LAST_VALUE(TransactionDate) OVER(PARTITION BY ProductID ORDER BY ProductID) as LastTransDate
FROM Production.TransactionHistory

## Proposition #8

In [ ]:
USE AdventureWorks2019;

## Proposition #9

In [ ]:
USE AdventureWorks2019;

## Proposition #10

In [ ]:
USE AdventureWorks2019;